In [ ]:
# ============================================================
# Cell 1 — Mount Drive, clone/pull repo, set Python path
# CST3990 Day 10 | Block C: F1+F2 Feature Extraction
# Student: MANJOO Ameera Najla | M01014463
# ============================================================
import os, sys

from google.colab import drive
drive.mount('/content/drive')

REPO_ROOT = '/content/CST-3990---Undergrad-Project'
DRIVE_ROOT = '/content/drive/MyDrive/CST3990'

if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/Mon-Amie-Geek/CST-3990---Undergrad-Project.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull origin main

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f'Working directory: {os.getcwd()}')
print(f'Python path includes: {REPO_ROOT}')

In [ ]:
# ============================================================
# Cell 2 — Install / verify required packages
# pandas and pyyaml should already be present in Colab.
# ============================================================
import importlib

for pkg, min_ver in [('pandas', '2.0'), ('yaml', '6.0')]:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  {pkg}: {ver} — OK')
    except ImportError:
        print(f'  {pkg}: NOT FOUND — installing...')
        if pkg == 'pandas':
            !pip install "pandas>=2.0" --quiet
        elif pkg == 'yaml':
            !pip install "pyyaml>=6.0" --quiet

print('Package check complete.')

In [ ]:
# ============================================================
# Cell 3 — Set all seeds (MUST run before any numpy/torch op)
# ============================================================
from src.seed_control import set_all_seeds
set_all_seeds()
print('All seeds set: torch=42, numpy=42, random=42, cudnn deterministic=True')

In [ ]:
# ============================================================
# Cell 4 — Pre-flight checks
# ============================================================
import json
import yaml

# Load metadata
with open('logs/metadata.json') as f:
    metadata = json.load(f)

train_seqs = metadata['split_seqs']['train']
val_seqs   = metadata['split_seqs']['val']
test_seqs  = metadata['split_seqs']['test']
all_seqs   = train_seqs + val_seqs + test_seqs

print('=== Split sequences ===')
print(f'  train ({len(train_seqs)}): {train_seqs}')
print(f'  val   ({len(val_seqs)}):   {val_seqs}')
print(f'  test  ({len(test_seqs)}):  {test_seqs}')
print(f'  total: {len(all_seqs)}')

# Split count assertions
assert len(train_seqs) == 7, f'Expected 7 train seqs, got {len(train_seqs)}'
assert len(val_seqs)   == 1, f'Expected 1 val seq, got {len(val_seqs)}'
assert len(test_seqs)  == 2, f'Expected 2 test seqs, got {len(test_seqs)}'
assert len(all_seqs)   == 10, (
    f'Expected 10 sequences total, got {len(all_seqs)}. '
    'Check logs/metadata.json split_seqs — all 10 UA-DETRAC sequences must be accounted for.'
)
assert len(set(all_seqs)) == 10, (
    f'Duplicate sequence IDs found across splits: '
    f'{[s for s in all_seqs if all_seqs.count(s) > 1]}. '
    'Each sequence must appear in exactly one split.'
)
print('Split count + uniqueness assertions passed.')

# Load block_b_results.json and confirm selected tracker
with open('logs/block_b_results.json') as f:
    block_b = json.load(f)

selected_tracker = None
for entry in block_b['trackers']:
    if entry.get('selected', False):
        selected_tracker = entry['tracker']
        break

assert selected_tracker is not None, (
    'No tracker marked selected in block_b_results.json["trackers"]. '
    'Day 9 tracker selection must be complete.'
)
print(f'Selected tracker from block_b_results.json: {selected_tracker}')

# Load config_blockC.yaml
assert os.path.exists('configs/config_blockC.yaml'), 'config_blockC.yaml not found'
with open('configs/config_blockC.yaml') as f:
    config = yaml.safe_load(f)

print(f'config_blockC.yaml loaded. tracker field: {config.get("tracker")}')

# Feature Contract Gate — must not be placeholder
assert config.get('tracker') not in (None, '', '<SELECTED_TRACKER_FROM_DAY9>'), (
    'config_blockC.yaml "tracker" field is not populated. '
    'Run populate_config_blockc_tracker() — confirm Day 9 is complete.'
)
print(f'Feature Contract Gate passed: tracker = "{config["tracker"]}"')
print('\nAll Cell 4 pre-flight checks PASSED.')

In [ ]:
# ============================================================
# Cell 5 — Layout detection
# ============================================================
from src.feature_extractor import detect_day7_layout, resolve_trajectory_path

base_dir = config.get('trajectory_base', 'logs/block_b/')
tracker  = config['tracker']

layout = detect_day7_layout(base_dir, tracker)
print(f'Detected trajectory layout: {layout}')

# Confirm at least one _final.json exists for test sequence MVI_20062
test_path = resolve_trajectory_path(base_dir, tracker, 'MVI_20062', layout)
assert os.path.exists(test_path), (
    f'_final.json not found at {test_path}. '
    'Run Day 9 post-processing on test sequences before Block C.'
)
print(f'MVI_20062 _final.json confirmed at: {test_path}')

# Show resolved paths for all sequences (some may be missing for non-test splits
# until full pipeline is run on train/val sequences in Colab)
print('\nResolved _final.json paths:')
for seq in all_seqs:
    p = resolve_trajectory_path(base_dir, tracker, seq, layout)
    exists = os.path.exists(p)
    status = 'OK' if exists else 'MISSING (will skip in generator)'
    print(f'  {seq}: {status}')

In [ ]:
# ============================================================
# Cell 6 — Warm-up verification
# ============================================================
import json
import pandas as pd
from src.feature_extractor import (
    flatten_tracks_dict, resolve_trajectory_path, validate_trajectory_file_metadata
)

demo_seq = 'MVI_20062'   # use test seq (guaranteed to exist)
demo_path = resolve_trajectory_path(base_dir, tracker, demo_seq, layout)

with open(demo_path) as f:
    traj_data = json.load(f)

entries = flatten_tracks_dict(traj_data)
if entries and 'bbox' in entries[0] and isinstance(entries[0]['bbox'], list):
    import copy
    for entry in entries:
        x1, y1, x2, y2 = entry.pop('bbox')
        entry['bbox_x1'], entry['bbox_y1'] = x1, y1
        entry['bbox_x2'], entry['bbox_y2'] = x2, y2

full_trajs = pd.DataFrame(entries)

fps = metadata['sequences'][demo_seq]['fps']
warmup_boundary = max(10, int(0.4 * fps))
print(f'{demo_seq}: fps={fps}, warmup_boundary={warmup_boundary}')
print(f'Full trajectory rows: {len(full_trajs)}')

print('\nFirst 5 rows BEFORE warm-up stripping (frame_idx column):')
print(full_trajs[['frame_idx', 'track_id', 'cx', 'cy', 'conf']].head())

trajs_stripped = full_trajs[full_trajs['frame_idx'] > warmup_boundary].copy()
print(f'\nRows AFTER warm-up stripping (>{warmup_boundary}): {len(trajs_stripped)}')
print('First 5 rows AFTER stripping:')
print(trajs_stripped[['frame_idx', 'track_id', 'cx', 'cy', 'conf']].head())

assert trajs_stripped['frame_idx'].min() > warmup_boundary, \
    f'Warm-up stripping failed: min frame_idx={trajs_stripped["frame_idx"].min()} <= {warmup_boundary}'

In [ ]:
# ============================================================
# Cell 7 — F1 feature extraction dry-run
# ============================================================
from src.feature_extractor import extract_f1_features

f1_df = extract_f1_features(trajs_stripped, fps, config)
print(f'F1 output shape: {f1_df.shape}')
print('F1 columns:', list(f1_df.columns))
print('\nHead:')
print(f1_df.head())

# Assertions
assert 'vehicle_count' in f1_df.columns, 'vehicle_count column missing from F1 output'
assert 'roi_occupancy' in f1_df.columns, 'roi_occupancy column missing from F1 output'
assert f1_df['vehicle_count'].isna().sum() == 0, 'NaN found in vehicle_count'
assert (f1_df['vehicle_count'] >= 1).all(), 'vehicle_count has zero-vehicle frames'
assert f1_df['roi_occupancy'].max() == 1.0, (
    f'roi_occupancy max should be 1.0, got {f1_df["roi_occupancy"].max():.4f}'
)
assert (f1_df['roi_occupancy'] >= 0.0).all() and (f1_df['roi_occupancy'] <= 1.0).all(), \
    'roi_occupancy out of [0,1] range'

print(f'\nF1 assertions passed.')
print(f'  vehicle_count range: [{f1_df["vehicle_count"].min()}, {f1_df["vehicle_count"].max()}]')
print(f'  roi_occupancy range: [{f1_df["roi_occupancy"].min():.4f}, {f1_df["roi_occupancy"].max():.4f}]')

In [ ]:
# ============================================================
# Cell 8 — F2 feature extraction dry-run
# ============================================================
from src.feature_extractor import extract_f2_features

f2_df = extract_f2_features(trajs_stripped, fps, config)
print(f'F2 output shape: {f2_df.shape}')
print('F2 columns:', list(f2_df.columns))
print('\nHead:')
print(f2_df[['frame_idx','track_id','velocity_norm','vel_px_sec','vel_px_sec_smooth','is_interpolated']].head(10))

# Assertions
assert 'vel_px_sec'        in f2_df.columns, 'vel_px_sec missing'
assert 'vel_px_sec_smooth' in f2_df.columns, 'vel_px_sec_smooth missing'
assert 'is_interpolated'   in f2_df.columns, 'is_interpolated missing'

# speed_window check: int(0.2 * 25.0) = 5
expected_window = int(0.2 * fps)
assert expected_window == 5, f'speed_window should be 5 at 25 FPS, got {expected_window}'
print(f'  speed_window = int(0.2 * {fps}) = {expected_window} frames — CORRECT')

# spot-check: velocity_norm=2.5 → vel_px_sec=62.5 at fps=25
sample_vn = f2_df['velocity_norm'].dropna().iloc[0]
sample_vps = f2_df['vel_px_sec'].dropna().iloc[0]
assert abs(sample_vps - sample_vn * fps) < 1e-9, (
    f'vel_px_sec formula wrong: {sample_vn} * {fps} != {sample_vps}'
)
print(f'  vel_px_sec formula spot-check passed: {sample_vn:.6f} * {fps} = {sample_vps:.4f}')

# vel_px_sec_smooth should not be NaN for any track (min_periods=1 ensures this)
nan_smooth = f2_df['vel_px_sec_smooth'].isna().sum()
print(f'  NaN in vel_px_sec_smooth: {nan_smooth} (should be 0 for non-NaN velocity_norm rows)')

# is_interpolated dtype
print(f'  is_interpolated dtype: {f2_df["is_interpolated"].dtype}')
print(f'  is_interpolated value counts:\n{f2_df["is_interpolated"].value_counts().to_dict()}')
print('\nF2 assertions passed.')

In [ ]:
# ============================================================
# Cell 9 — Full generator run over all 10 sequences
# ============================================================
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')

from src.feature_extractor import clip_feature_generator, populate_config_blockc_tracker

# Re-load config fresh (populate tracker if needed)
with open('configs/config_blockC.yaml') as f:
    config = yaml.safe_load(f)
config = populate_config_blockc_tracker(config)

print(f'Running generator over {len(all_seqs)} sequences with tracker={config["tracker"]}...')
completed_clips = []
for clip_id in clip_feature_generator(all_seqs, metadata, config):
    completed_clips.append(clip_id)
    print(f'  [DONE] {clip_id}')

print(f'\nGenerator complete. {len(completed_clips)}/{len(all_seqs)} clips processed.')

# Verify output CSVs
features_dir = config.get('features_dir', 'logs/block_c/')
csv_files = [f for f in os.listdir(features_dir) if f.endswith('_features.csv')]
print(f'CSV files in {features_dir}: {sorted(csv_files)}')

# At least the 2 test-split CSVs must exist
for seq in test_seqs:
    p = os.path.join(features_dir, f'{seq}_features.csv')
    assert os.path.exists(p), f'Missing features CSV for test seq: {seq}'
print('Test-split CSVs confirmed.')

In [ ]:
# ============================================================
# Cell 10 — Schema validation on MVI_20062
# ============================================================
from src.feature_extractor import FEATURE_SCHEMA_COLUMN_ORDER

demo_csv = os.path.join(config.get('features_dir', 'logs/block_c/'), 'MVI_20062_features.csv')
df = pd.read_csv(demo_csv)

print(f'Loaded: {demo_csv}')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print('\nDtypes:')
print(df.dtypes)

# Assert all schema columns present in correct order
for col in FEATURE_SCHEMA_COLUMN_ORDER:
    assert col in df.columns, f'Schema column missing: {col}'
# Assert column ORDER matches schema order
actual_schema_cols = [c for c in df.columns if c in FEATURE_SCHEMA_COLUMN_ORDER]
expected_schema_cols = [c for c in FEATURE_SCHEMA_COLUMN_ORDER if c in df.columns]
assert actual_schema_cols == expected_schema_cols, (
    f'Column order mismatch.\n  Expected: {expected_schema_cols}\n  Got: {actual_schema_cols}'
)
print('\nAll 19 schema columns present in correct order — schema assertion PASSED.')

print('\nis_interpolated value counts:')
print(df['is_interpolated'].value_counts().to_dict())

print('\nsplit value counts:')
print(df['split'].value_counts().to_dict())

assert set(df['split'].unique()).issubset({'train', 'val', 'test'}), \
    'Invalid split values found'
print('\nSplit values valid (no "unknown").')

In [ ]:
# ============================================================
# Cell 11 — Spot-check interpolated frames: conf must be -1.0
# ============================================================
total_interp = 0
issues = []

features_dir = config.get('features_dir', 'logs/block_c/')
for csv_file in sorted(os.listdir(features_dir)):
    if not csv_file.endswith('_features.csv'):
        continue
    df_c = pd.read_csv(os.path.join(features_dir, csv_file))
    interp_rows = df_c[df_c['is_interpolated'] == True]
    n_interp = len(interp_rows)
    total_interp += n_interp
    if n_interp > 0:
        bad = interp_rows[interp_rows['conf'] != -1.0]
        if len(bad) > 0:
            issues.append(f'  {csv_file}: {len(bad)} interpolated rows with conf != -1.0')
        print(f'  {csv_file}: {n_interp} interpolated rows — sentinel check: {"FAIL" if len(bad) else "OK"}')
    else:
        print(f'  {csv_file}: 0 interpolated rows')

print(f'\nTotal interpolated frames across all CSVs: {total_interp}')
if issues:
    print('SENTINEL ISSUES:')
    for i in issues:
        print(i)
else:
    print('All interpolated rows have conf=-1.0 sentinel — PASSED.')

In [ ]:
# ============================================================
# Cell 12 — Print final state of config_blockC.yaml
# ============================================================
import yaml

print('=== configs/config_blockC.yaml (final Day 10 state) ===')
with open('configs/config_blockC.yaml') as f:
    raw = f.read()
print(raw)

# Confirm key fields
cfg = yaml.safe_load(raw)
assert cfg.get('tracker') not in (None, '', '<SELECTED_TRACKER_FROM_DAY9>'), \
    'tracker placeholder still present — populate_config_blockc_tracker() must be run'
assert cfg.get('roi_policy') == 'full_frame',    'roi_policy: full_frame missing'
assert cfg.get('f2', {}).get('speed_window_sec') == 0.2, 'speed_window_sec: 0.2 missing'
assert cfg.get('scaler', {}).get('fit_split') == 'train', 'scaler.fit_split: train missing'
assert cfg.get('feature_schema') == 'configs/feature_schema.json', 'feature_schema path wrong'
assert cfg.get('f3', {}).get('inter_vehicle_dist_norm') == False, 'f3 not stubbed correctly'
print('\nAll config_blockC.yaml field assertions PASSED.')

In [ ]:
# ============================================================
# Cell 13 — Drive backup
# ============================================================
import shutil

# Backup all block_c CSVs
drive_block_c = os.path.join(DRIVE_ROOT, 'logs', 'block_c')
os.makedirs(drive_block_c, exist_ok=True)

features_dir = config.get('features_dir', 'logs/block_c/')
for f in os.listdir(features_dir):
    if f.endswith('.csv'):
        src = os.path.join(features_dir, f)
        dst = os.path.join(drive_block_c, f)
        shutil.copy2(src, dst)
        print(f'  Backed up: {f}')

# Backup updated configs
drive_configs = os.path.join(DRIVE_ROOT, 'configs')
os.makedirs(drive_configs, exist_ok=True)
for cfg_file in ['config_blockC.yaml', 'feature_schema.json']:
    shutil.copy2(f'configs/{cfg_file}', os.path.join(drive_configs, cfg_file))
    print(f'  Backed up: configs/{cfg_file}')

print('\nDrive backup complete.')

In [ ]:
# ============================================================
# Cell 14 — GitHub commit and push
# ============================================================
!git config user.email "ameera@example.com"
!git config user.name "Mon-Amie-Geek"

!git add src/feature_extractor.py
!git add configs/config_blockC.yaml
!git add configs/feature_schema.json
!git add src/pipeline_controller.py
!git add requirements.txt
!git add notebooks/day10_block_c_features.ipynb

!git status

!git commit -m "Day 10: Block C F1+F2 features + generator loader + feature_schema.json + config_blockC.yaml complete"

!git push origin main
print('GitHub push complete.')